# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OjaswiGautam/FlyrankAI/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Lane 3 (Structured Content Archetype Clustering) is fundamentally a clustering task — specifically, unsupervised partitioning of content items into groups based on similarity across behavioral and structural features (visibility, engagement, efficiency, content shape).


Why clustering specifically fits this lane: the underlying question — "What recurring performance archetypes exist in the inventory?"

— has no ground truth to supervise against:

 I don't know in advance how many archetypes exist or what defines them; that's the thing being discovered. That's the definition of an unsupervised problem, and K-Means (or a comparable partitioning method) is the standard tool for it.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Lane 3 has no target, this is a genuinely unsupervised task, so there is no single column I'm predicting and no proxy label standing in for one.
What clustering uses instead is a feature space, not a target: a set of behavioral and structural signals (log-scaled impressions/sessions, CTR, avg_position, engagement_rate, scroll_rate, content_age_days, word_count, content_type/main_intent, trend_direction) that every content item has a value for. The algorithm groups items by proximity in that space and there's nothing being "predicted" and nothing to be right or wrong against.


The one place a "label" appears in this lane, and why it's not a target: after clustering, I assign each resulting cluster a human-readable name and an action (protect/improve/rewrite/merge/prune/monitor) based on inspecting real rows. That's descriptive labeling of an output, applied post-hoc. It's not something the algorithm is trained to predict, and it must never be fed back in as a feature or confused with a supervised target.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary metric: Mean Silhouette Score on the scaled/log-transformed feature space.
This stays my answer to "one metric you can defend" because the notebook section specifically asks for a single defensible number — and silhouette is the right one to lead with: it needs no ground-truth labels, it captures both cohesion (how tight a cluster is) and separation (how distinct it is from its nearest neighbor), and it's the metric a non-technical reviewer can most easily be walked through. Target: ≥0.25–0.30, checked for stability across a K range (roughly K=4–8) rather than chasing the single highest value, since silhouette can be gamed by artificially small K.
Where the fuller framework you shared fits — this is the correction I needed: silhouette alone is the headline number, but selecting and validating the final clustering solution is not a single-metric decision. Per the framework, I'll carry a support stack alongside it:

*   Inertia — elbow-method diagnostic only, for narrowing candidate K values (mechanically decreases with K, so never a standalone decision rule)

*   Calinski-Harabasz — supporting separation/compactness check, interpreted alongside silhouette rather than on its own (it's biased toward higher K)



*   Davies-Bouldin — supporting nearest-neighbor cluster separation check (lower = better)

*   Cluster size distribution — guards against one dominant cluster + several throwaway tiny ones

*   Stability across seeds and subsamples (via Adjusted Rand Index) — confirms the archetypes are real structure, not initialization noise

*   Centroid/profile separation, interpretability, actionability, driver analysis, outlier/preprocessing sensitivity, leakage/circularity audit, recommendation sanity checks — all correctness and honesty checks that happen once a K is chosen, before I trust or publish the archetypes

Reconciling this with "one metric": silhouette is the number I'd put in a headline sentence ("clusters achieved a mean silhouette of X"). Everything else in the framework is what proves that number is trustworthy rather than a statistical accident — that validation work belongs to later notebooks (w04/w05, the model + validation stages), not this framing notebook. I'll flag now that I'll need those steps explicitly built in when we get there, not skipped.

Final selection rule I'll carry forward (matches the document): don't pick K on silhouette alone — pick the K where silhouette, inertia's elbow, Calinski-Harabasz, Davies-Bouldin, stability, cluster-size sanity, and interpretability jointly point to the same answer.


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [10]:
url = 'https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv'
!wget -q -O content_refresh_anonymized.csv {url}

In [11]:
from pathlib import Path
import pandas as pd

lane3_csv_path = Path("content_refresh_anonymized.csv")

df_lane3_source = pd.read_csv(lane3_csv_path)

forbidden_product_decision_fields = {
    "health_score", "priority_score", "action_type",
    "recommended_action", "recommendation", "cluster", "archetype",
}

lane3_unit_id_cols = ["content_id", "client_id"]

lane3_observable_signal_cols = [
    "content_type", "main_intent", "search_volume", "competition", "competition_level", "cpc",
    "word_count", "char_count", "provider_used", "model_used", "content_age_days", "age_tier",
    "age_tier_order", "days_since_last_update", "freshness_tier", "word_count_tier", "char_count_tier",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d", "engaged_sessions_90d",
    "ai_sessions_90d", "scroll_events_90d", "days_with_impressions", "days_with_sessions",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d", "impressions_prev_30d",
    "clicks_prev_30d", "sessions_prev_30d", "ctr", "avg_position", "engagement_rate", "scroll_rate",
    "ai_traffic_pct", "impression_tier", "position_tier", "trend_direction", "trend_pct",
]

available_lane3_cols = [c for c in lane3_unit_id_cols + lane3_observable_signal_cols
                         if c in df_lane3_source.columns and c not in forbidden_product_decision_fields]
forbidden_fields_selected = sorted(forbidden_product_decision_fields.intersection(available_lane3_cols))
if forbidden_fields_selected:
    raise ValueError("Forbidden product-decision fields selected: " + ", ".join(forbidden_fields_selected))

lane3_unit_df = df_lane3_source.loc[:, available_lane3_cols].copy()

print("Source shape:", df_lane3_source.shape)
print("Lane 3 unit dataframe shape:", lane3_unit_df.shape)
print("unique content_id:", lane3_unit_df["content_id"].nunique())

lane3_preview_cols = [
    "content_id", "client_id", "content_type", "main_intent",
    "impressions_90d", "clicks_90d", "sessions_90d", "engaged_sessions_90d",
    "ctr", "avg_position", "engagement_rate", "content_age_days",
    "freshness_tier", "trend_direction", "trend_pct",
]
lane3_preview_cols = [c for c in lane3_preview_cols if c in lane3_unit_df.columns]
lane3_unit_df[lane3_preview_cols].head(5)

Source shape: (30000, 44)
Lane 3 unit dataframe shape: (30000, 44)
unique content_id: 30000


,content_id,client_id,content_type,main_intent,impressions_90d,clicks_90d,sessions_90d,engaged_sessions_90d,ctr,avg_position,engagement_rate,content_age_days,freshness_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3803,29,17,1,0.76,10.6,5.88,187,0-30,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,15320,7,9,0,0.05,20.3,0.00,445,0-30,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,12581,11,11,0,0.09,36.5,0.00,141,0-30,down,-60.9
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,11751,58,78,1,0.49,6.2,1.28,463,0-30,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,19140,24,145,0,0.13,44.0,0.00,263,0-30,down,-34.7


One row in this dataframe = one content item (one page), keyed by content_id. This is confirmed, not assumed: loading the starter dataset and checking content_id.nunique() against the row count gives 30,000 unique IDs across 30,000 rows — a verified 1:1 grain.
This is the correct unit for Lane 3 because archetypes are a page-level behavioral pattern. A daily grain (the source fact table) would split one page into ~90 competing snapshots instead of one profile. A client grain would blend together many pages with very different behavior. A query grain (content×query) describes something else — how a page performs per search term — which is a possible feature source later, not the clustering unit itself.
The dataframe pulls in all 42 legitimate observable columns — content metadata, freshness, volume, engagement, recent-window trend inputs, and tier fields — rather than a hand-picked subset, since the unit of analysis should reflect everything legitimately available at that grain. Seven known circular fields (health_score, priority_score, action_type, recommended_action, recommendation, cluster, archetype) are excluded, enforced by a check that raises an error if any are ever selected — not just stated in prose. None of them happen to exist in this starter CSV, but the same guard matters more once the full warehouse tables are in play.
The preview below shows five real rows at this grain, so the shape of a single unit of analysis is visible, not just described.
One data-quality issue this surfaced: word_count has nulls in some rows — will need explicit handling (impute, drop, or fall back to word_count_tier) during feature engineering.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule for archetypes would look like nested if-statements — e.g., "if impressions_90d is high AND ctr is low AND trend_direction is down, label it X." ML is better suited here than a fixed rule — not because ML is categorically superior, but because the goal of this lane is to discover unknown behavioral archetypes from multiple interacting signals, rather than apply a known, pre-defined boundary. A fixed rule remains the right tool when the decision boundary is already known and interpretability is the top priority (which is closer to Lane 2's baseline-scoring approach) — but that's not this task.
1. The feature space is high-dimensional, and a nested rule can only carve axis-aligned boxes. Even restricting to core behavioral signals (impressions, CTR, avg_position, engagement_rate, content_age_days, word_count, trend_pct), a rule-based approach needs a human to hand-pick thresholds across 6+ continuous variables. A nested rule creates axis-aligned rectangular buckets — it can only split on one variable at a time. K-Means, more precisely, groups pages by Euclidean proximity across the full scaled feature space, forming convex, distance-based clusters — not arbitrarily-shaped ones, but still able to capture interactions between variables that a rectangular rule structurally cannot.
2. Existing tier columns are useful for interpreting clusters afterward, not necessarily as clustering features themselves. Columns like impression_tier, position_tier, age_tier, and trend_direction (confirmed: 4 impression tiers, 5 position tiers, 5 trend directions) are valuable for describing and naming clusters once they're found — e.g., "this cluster is 80% page_1 position tier." But feeding coarse categorical tiers directly into K-Means as primary features would throw away the finer-grained continuous information already available in the underlying numeric columns (avg_position, impressions_90d, etc.). I'll use the tiers as a validation and labeling aid against cluster output, and the continuous fields as the actual clustering inputs.
3. A rule-based approach naively enumerating tier combinations produces mostly-empty buckets. 4 × 5 × 6 × 5 × 5 tier combinations across just five categorical fields is hundreds of buckets, most sparsely populated or empty. Clustering instead lets the actual joint distribution of the data decide which combinations are common enough to be real archetypes.
4. Thresholds would be arbitrary unless tied to a validated outcome or business constraint. Any fixed rule requires someone to decide, by hand, what counts as "high" impressions or "low" CTR — and scale varies enormously across this data (word_count spans 8 to 9,546; impressions span multiple orders of magnitude). Without a labeled outcome or an explicit business rule to justify a cutoff, that threshold is just an eyeballed guess. Clustering derives group boundaries from the data's own structure instead.
5. K-Means only works after appropriate preprocessing — this is part of why it's the right tool, not a caveat against it. Since K-Means is distance-based, skewed variables like impressions_90d, clicks_90d, sessions_90d, and word_count need log-transforming before use, or their long tails would dominate the distance calculation and drown out every other signal. All numeric features then need scaling (StandardScaler) so no single feature dominates just because of its raw units. Circular fields (health_score, priority_score, action_type, etc.) are excluded entirely, per the leakage guard already built into the Section 4 code.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.